# 멀티모달 LLM - 실습 코드 1: LLaVA 스타일 비전-언어 모델 구현

- Tutorial ID: `expand-multimodal-llm`
- Tutorial: 멀티모달 LLM
- Section ID: `expand-multimodal-llm-code-1`
- Section: 실습 코드 1: LLaVA 스타일 비전-언어 모델 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: LLaVA 스타일 비전-언어 모델 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
from transformers import CLIPVisionModel, CLIPImageProcessor
from transformers import LlamaForCausalLM, LlamaTokenizer

# ── LLaVA 스타일 멀티모달 모델 ──
class LLaVAStyleModel(nn.Module):
    """LLaVA: Visual Instruction Tuning 아키텍처 구현"""
        def __init__(self, vision_model_name, llm_model_name, projector_dim=768):
        super().__init__()
        # 비전 인코더 (CLIP ViT)
        self.vision_encoder = CLIPVisionModel.from_pretrained(vision_model_name)
                for param in self.vision_encoder.parameters():
            param.requires_grad = False  # Vision Encoder冻结
        
        # 언어 모델 (LLM)
        self.llm = LlamaForCausalLM.from_pretrained(llm_model_name)
        
        # MLP Projector: Vision → Language 공간
        vision_hidden = self.vision_encoder.config.hidden_size  # 1024
        llm_hidden = self.llm.config.hidden_size               # 4096
        
        self.projector = nn.Sequential(
            nn.Linear(vision_hidden, projector_dim),
            nn.GELU(),
            nn.Linear(projector_dim, llm_hidden),
        )
        
        self.image_processor = CLIPImageProcessor.from_pretrained(vision_model_name)
        self.tokenizer = LlamaTokenizer.from_pretrained(llm_model_name)
    
        def encode_image(self, pixel_values):
        """이미지 → 시각 토큰"""
        vision_outputs = self.vision_encoder(pixel_values)
        # CLS 토큰 + 패치 토큰 사용
        image_features = vision_outputs.last_hidden_state  # (batch, num_patches, vision_dim)
        # Projector를 통해 LLM 입력 공간으로 변환
        visual_tokens = self.projector(image_features)      # (batch, num_patches, llm_dim)
        return visual_tokens
    
        def forward(self, pixel_values, input_ids, labels=None):
        """멀티모달 Forward Pass"""
        # 1. 이미지 인코딩
        visual_tokens = self.encode_image(pixel_values)  # (1, 576, 4096)
        
        # 2. 텍스트 임베딩
        text_embeds = self.llm.get_input_embeddings()(input_ids)  # (1, seq, 4096)
        
        # 3. [IMAGE] 토큰 위치에 시각 토큰 삽입
        image_token_id = self.tokenizer.convert_tokens_to_ids("<image>")
                image_mask = (input_ids == image_token_id)
        
        # 시각 토큰으로 교체
        combined_embeds = text_embeds.clone()
                for b in range(input_ids.size(0)):
                        img_positions = image_mask[b].nonzero(as_tuple=True)[0]
            if len(img_positions) > 0:
                # <image> 토큰 수만큼 시각 토큰 삽입
                n_visual = visual_tokens.size(1)
                                for i, pos in enumerate(img_positions[:n_visual]):
                    combined_embeds[b, pos] = visual_tokens[b, i]
        
        # 4. LLM 처리
        outputs = self.llm(
            inputs_embeds=combined_embeds,
            labels=labels,
        )
        return outputs

# ── 추론 함수 ──
def multimodal_generate(model, image, prompt, max_new_tokens=200):
    """이미지 + 텍스트 프롬프트로 응답 생성"""
    # 이미지 전처리
    pixel_values = model.image_processor(
        images=image, return_tensors="pt"
    ).pixel_values
    
    # 프롬프트에 <image> 토큰 포함
    formatted_prompt = f"<image>\nUSER: {prompt}\nASSISTANT:"
    input_ids = model.tokenizer(
        formatted_prompt, return_tensors="pt"
    ).input_ids
    
    # 생성
    with torch.no_grad():
        outputs = model.llm.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
                        temperature=0.7,
            do_sample=True,
        )
    
    return model.tokenizer.decode(outputs[0], skip_special_tokens=True)

# 모델 초기화 (예시)
print("LLaVA 스타일 멀티모달 모델 아키텍처 준비 완료")
print("Vision Encoder (CLIP) → MLP Projector → LLM (Llama)")